# Exercises — Chapter 09: Financial NLP and Sentiment Analysis

Complete the exercises from Lecture 09.

In [ ]:
# Your code here

## Data Lab — SEC EDGAR

Test whether text sentiment in annual filings is associated with stock returns. All financial data comes from EDGAR (filings) and yfinance (prices).

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course your-email@example.com"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

LM_POS = {"profitable","strong","growth","improved","exceeded","innovative",
           "efficient","leading","record","gained","successful","robust",
           "increased","advancing","outperformed","benefit","enhanced","positive","achieved","favourable"}
LM_NEG = {"risk","loss","uncertain","decline","adverse","default","impair",
           "volatile","litigation","breach","failed","reduced","weakness",
           "debt","distress","penalty","violation","exposure","downturn","writedown"}


### Exercise [B]: Loughran–McDonald Sentiment Scoring

In [ ]:
# --- Exercise [B]: LM Sentiment Scoring ---
TICKERS_09 = ["AAPL","MSFT","JPM","XOM"]

def lm_sentiment(text):
    words = re.findall(r"[a-z]+", text.lower())
    n     = max(len(words), 1)
    p_cnt = sum(1 for w in words if w in LM_POS) / n
    n_cnt = sum(1 for w in words if w in LM_NEG) / n
    net   = (p_cnt - n_cnt) / (p_cnt + n_cnt + 1e-9)
    return {"pos": p_cnt, "neg": n_cnt, "net": net}

rows_b = []
for t in TICKERS_09:
    html = fetch_10k_html(t)
    mda  = extract_mda(html)
    s    = lm_sentiment(mda)
    rows_b.append({"ticker": t, **s})
    print(f"{t}: pos={s['pos']:.4f}, neg={s['neg']:.4f}, net={s['net']:.4f}")

df_sent = pd.DataFrame(rows_b).set_index("ticker")
fig, ax = plt.subplots(figsize=(7,4))
colors  = ["green" if v >= 0 else "tomato" for v in df_sent["net"]]
ax.barh(df_sent.index, df_sent["net"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Net LM Sentiment"); ax.set_title("MD&A Net Sentiment (LM Lexicon)")
plt.tight_layout(); plt.show()

### Exercise [I]: Sentiment vs. Filing-Day Return

In [ ]:
# --- Exercise [I]: Sentiment vs. Filing-Day Return ---
TICKERS_09I = ["AAPL","MSFT","GOOGL","JPM","BAC","XOM","JNJ","PFE","T","GE"]

rows_i = []
for t in TICKERS_09I:
    try:
        cik  = get_cik(t)
        subs = get_submissions(cik)
        f    = subs["filings"]["recent"]
        idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
        date = f["filingDate"][idx]
        html = fetch_10k_html(t)
        mda  = extract_mda(html)
        s    = lm_sentiment(mda)["net"]
        try:
            import yfinance as yf
            t0 = pd.Timestamp(date); t1 = t0 + pd.Timedelta(days=5)
            hist = yf.download(t, start=t0, end=t1, progress=False, auto_adjust=True)["Close"]
            ret  = float(hist.iloc[1]/hist.iloc[0]-1) if len(hist)>=2 else None
        except: ret = None
        if ret is not None:
            rows_i.append({"ticker":t,"sentiment":s,"return":ret})
            print(f"{t}: sent={s:.3f}, ret={ret:.3f}")
    except Exception as e:
        print(f"{t}: {e}")

df_i = pd.DataFrame(rows_i)
if len(df_i) >= 3:
    fig, ax = plt.subplots(figsize=(7,5))
    ax.scatter(df_i["sentiment"], df_i["return"]*100, s=80, color="steelblue")
    for _, r in df_i.iterrows(): ax.annotate(r["ticker"],(r["sentiment"],r["return"]*100),fontsize=8)
    corr = np.corrcoef(df_i["sentiment"], df_i["return"])[0,1]
    ax.set_xlabel("Net LM Sentiment"); ax.set_ylabel("Filing-Day Return (%)")
    ax.set_title(f"Sentiment vs. Filing-Day Return  (r={corr:.2f})")
    plt.tight_layout(); plt.show()
    print(f"Pearson r = {corr:.3f}")

### Exercise [A]: Filing-Date Event Study

In [ ]:
# --- Exercise [A]: Event Study ---
TICKERS_09A = ["AAPL","MSFT","GOOGL","JPM","BAC","XOM","JNJ","PFE","GE","F"]
WIN = (-5, 10)

cars = {"positive": [], "negative": []}
for t in TICKERS_09A:
    try:
        cik  = get_cik(t)
        subs = get_submissions(cik)
        f    = subs["filings"]["recent"]
        idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
        date = pd.Timestamp(f["filingDate"][idx])
        html = fetch_10k_html(t)
        mda  = extract_mda(html)
        s    = lm_sentiment(mda)["net"]
        try:
            import yfinance as yf
            t0 = date + pd.Timedelta(days=WIN[0]-2)
            t1 = date + pd.Timedelta(days=WIN[1]+5)
            stock = yf.download(t,    start=t0, end=t1, progress=False, auto_adjust=True)["Close"]
            mkt   = yf.download("^GSPC", start=t0, end=t1, progress=False, auto_adjust=True)["Close"]
            if len(stock)<5 or len(mkt)<5: continue
            common = stock.index.intersection(mkt.index)
            # find event date index
            ev_idx = common.searchsorted(date)
            if ev_idx < 5 or ev_idx > len(common)-WIN[1]-1: continue
            window = common[ev_idx+WIN[0]:ev_idx+WIN[1]+1]
            if len(window) < 10: continue
            s_ret = stock.reindex(window).pct_change().fillna(0).values
            m_ret = mkt.reindex(window).pct_change().fillna(0).values
            car   = np.cumsum(s_ret - m_ret)
            grp   = "positive" if s > 0 else "negative"
            cars[grp].append(car)
        except Exception as e2:
            print(f"{t} yf: {e2}")
    except Exception as e:
        print(f"{t}: {e}")

fig, ax = plt.subplots(figsize=(9,5))
for grp, c in [("positive","green"),("negative","tomato")]:
    if cars[grp]:
        arr = np.array([x[:min(16,len(x))] for x in cars[grp]])
        mean_car = arr.mean(0)
        days = range(WIN[0], WIN[0]+len(mean_car))
        ax.plot(days, mean_car*100, color=c, linewidth=2, label=f"{grp} sentiment (n={len(cars[grp])})")
ax.axvline(0, color="black", linestyle="--"); ax.axhline(0, color="gray", linewidth=0.5)
ax.set_xlabel("Days relative to filing"); ax.set_ylabel("Mean CAR (%)")
ax.set_title("Cumulative Abnormal Return Around 10-K Filing Date")
ax.legend(); plt.tight_layout(); plt.show()